# Phát hiện ảnh gần trùng và suy luận nhóm PKU-Market-PCB

Notebook này là artifact chính của Tuần 3. Notebook chỉ đọc toàn bộ ảnh, kể cả ảnh trong thư mục lồng nhau, dưới `data/raw/PKU-Market-PCB(Data enhanced version)`; không sửa dữ liệu gốc. Mỗi ảnh được tính perceptual hash (`imagehash.phash`), mọi cặp có khoảng cách Hamming không quá 5 được xuất để phục vụ kiểm tra rò rỉ dữ liệu khi chia tập.

Tệp nhóm cuối cùng là `data/splits/groups.csv`, với đúng bốn cột yêu cầu. Ưu tiên của `group_id` là tiền tố số đầu tiên trong tên tệp, ví dụ `01_missing_hole_01.jpg` nhận nhóm `01`. Chỉ ảnh không có tiền tố này mới dùng nhóm thành phần liên thông từ đồ thị các cặp phash gần nhau.

## Phạm vi và tính tái lập

Đường dẫn trong các CSV luôn tương đối với thư mục gốc dự án và dùng dấu gạch chéo `/`, vì vậy không phụ thuộc máy đang chạy. Danh sách ảnh được sắp xếp trước khi băm, so sánh và đánh số thành phần liên thông để kết quả ổn định. Phép so sánh xét từng cặp ảnh khác nhau đúng một lần; khoảng cách Hamming là số bit khác nhau giữa hai phash 64-bit.

Ngưỡng 5 là quy tắc phát hiện ứng viên gần trùng, không khẳng định hai ảnh hoàn toàn giống nhau. Thành phần liên thông dùng quan hệ bắc cầu: nếu A gần B và B gần C thì cả ba thuộc cùng một cụm phash. Quy tắc này chỉ là phương án dự phòng khi tên tệp không cung cấp tiền tố số.

In [1]:
import csv
import re
from pathlib import Path

import imagehash
from PIL import Image


HAU_TO_ANH = {'.bmp', '.jpeg', '.jpg', '.png', '.tif', '.tiff', '.webp'}
NGUONG_HAMMING = 5
COT_NHOM = ['image_path', 'group_id', 'phash', 'derived_by']
COT_CAP = ['image_path_a', 'image_path_b', 'phash_a', 'phash_b', 'hamming_distance']


def tim_goc_du_an() -> Path:
    for ung_vien in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (ung_vien / 'data').is_dir() and (ung_vien / 'scripts').is_dir():
            return ung_vien
    raise RuntimeError('Không tìm thấy thư mục gốc dự án.')


GOC_DU_AN = tim_goc_du_an()
THU_MUC_PKU = GOC_DU_AN / 'data/raw/PKU-Market-PCB(Data enhanced version)'
DUONG_DAN_NHOM = GOC_DU_AN / 'data/splits/groups.csv'
DUONG_DAN_CAP = GOC_DU_AN / 'results/pku_near_duplicate_pairs.csv'
DUONG_DAN_AUDIT = GOC_DU_AN / 'docs/data_audit.md'

if not THU_MUC_PKU.is_dir():
    raise FileNotFoundError(f'Không tìm thấy dataset PKU: {THU_MUC_PKU}')


## Tính perceptual hash cho toàn bộ ảnh

Ảnh được nhận diện theo phần mở rộng thông dụng, không giả định cấu trúc `train`, `test` hay `images`; vì vậy ảnh ở mọi thư mục con đều được đưa vào. `Image.open` được dùng trong context manager và `load()` đọc dữ liệu trước khi tệp đóng. Chuỗi phash hệ cơ số mười sáu có độ dài 16 ký tự được giữ trong CSV để có thể kiểm tra độc lập.

In [2]:
def duong_dan_tuong_doi(duong_dan: Path) -> str:
    return duong_dan.relative_to(GOC_DU_AN).as_posix()


duong_dan_anh = sorted(
    (duong_dan for duong_dan in THU_MUC_PKU.rglob('*') if duong_dan.is_file() and duong_dan.suffix.lower() in HAU_TO_ANH),
    key=duong_dan_tuong_doi,
)
if not duong_dan_anh:
    raise RuntimeError(f'Không tìm thấy ảnh trong {THU_MUC_PKU}.')

ban_ghi = []
for duong_dan in duong_dan_anh:
    with Image.open(duong_dan) as anh:
        anh.load()
        gia_tri_phash = imagehash.phash(anh)
    ban_ghi.append({
        'image_path': duong_dan_tuong_doi(duong_dan),
        'phash': str(gia_tri_phash),
        'phash_int': int(str(gia_tri_phash), 16),
        'filename_prefix': re.match(r'^(\d+)', duong_dan.name).group(1) if re.match(r'^(\d+)', duong_dan.name) else None,
    })

print(f'Đã tính phash cho {len(ban_ghi):,} ảnh PKU.')


Đã tính phash cho 4,837 ảnh PKU.


## Tìm cặp gần trùng và thành phần liên thông

Mọi tổ hợp hai ảnh được so sánh bằng XOR của số nguyên phash, rồi đếm số bit khác nhau. Với mỗi cặp đạt ngưỡng, notebook lưu một hàng vào CSV và hợp nhất hai chỉ số trong cấu trúc union-find. Sau bước này, `goc()` trả về đại diện của thành phần liên thông chứa ảnh; đại diện nhỏ nhất theo thứ tự ảnh bảo đảm tên cụm dự phòng tái lập.

In [3]:
cha = list(range(len(ban_ghi)))


def goc(chi_so: int) -> int:
    while cha[chi_so] != chi_so:
        cha[chi_so] = cha[cha[chi_so]]
        chi_so = cha[chi_so]
    return chi_so


def hop_nhat(a: int, b: int) -> None:
    goc_a, goc_b = goc(a), goc(b)
    if goc_a != goc_b:
        cha[max(goc_a, goc_b)] = min(goc_a, goc_b)


cap_gan_trung = []
for chi_so_a, ban_ghi_a in enumerate(ban_ghi):
    for chi_so_b in range(chi_so_a + 1, len(ban_ghi)):
        ban_ghi_b = ban_ghi[chi_so_b]
        khoang_cach = (ban_ghi_a['phash_int'] ^ ban_ghi_b['phash_int']).bit_count()
        if khoang_cach <= NGUONG_HAMMING:
            cap_gan_trung.append({
                'image_path_a': ban_ghi_a['image_path'],
                'image_path_b': ban_ghi_b['image_path'],
                'phash_a': ban_ghi_a['phash'],
                'phash_b': ban_ghi_b['phash'],
                'hamming_distance': khoang_cach,
            })
            hop_nhat(chi_so_a, chi_so_b)

thanh_phan = {}
for chi_so in range(len(ban_ghi)):
    thanh_phan.setdefault(goc(chi_so), []).append(chi_so)

# Đánh số theo ảnh nhỏ nhất trong từng thành phần để nhóm dự phòng ổn định giữa các lần chạy.
thu_tu_thanh_phan = sorted(thanh_phan, key=lambda dai_dien: ban_ghi[min(thanh_phan[dai_dien])]['image_path'])
ma_cum = {dai_dien: f'phash_cluster_{so_thu_tu:04d}' for so_thu_tu, dai_dien in enumerate(thu_tu_thanh_phan, start=1)}

print(f'Đã tìm thấy {len(cap_gan_trung):,} cặp gần trùng trong {len(thanh_phan):,} thành phần phash.')


Đã tìm thấy 217,179 cặp gần trùng trong 1,923 thành phần phash.


## Suy luận và xuất nhóm

Mỗi ảnh có đúng một hàng. Nếu tên tệp bắt đầu bằng một hay nhiều chữ số, giá trị chữ số đó được giữ nguyên làm `group_id` và `derived_by` là `filename_prefix`. Nếu không có tiền tố, `group_id` là mã thành phần liên thông phash và `derived_by` là `phash_cluster`. Không dùng phash để ghi đè nhóm suy ra từ tên, kể cả khi ảnh có cạnh gần trùng, vì quy tắc ưu tiên này làm nguồn gốc của từng nhóm rõ ràng.

In [4]:
hang_nhom = []
for chi_so, ban_ghi_anh in enumerate(ban_ghi):
    tien_to = ban_ghi_anh['filename_prefix']
    if tien_to is not None:
        group_id, derived_by = tien_to, 'filename_prefix'
    else:
        group_id, derived_by = ma_cum[goc(chi_so)], 'phash_cluster'
    hang_nhom.append({
        'image_path': ban_ghi_anh['image_path'],
        'group_id': group_id,
        'phash': ban_ghi_anh['phash'],
        'derived_by': derived_by,
    })

DUONG_DAN_NHOM.parent.mkdir(parents=True, exist_ok=True)
with DUONG_DAN_NHOM.open('w', newline='', encoding='utf-8') as tep_csv:
    writer = csv.DictWriter(tep_csv, fieldnames=COT_NHOM)
    writer.writeheader()
    writer.writerows(hang_nhom)

DUONG_DAN_CAP.parent.mkdir(parents=True, exist_ok=True)
with DUONG_DAN_CAP.open('w', newline='', encoding='utf-8') as tep_csv:
    writer = csv.DictWriter(tep_csv, fieldnames=COT_CAP)
    writer.writeheader()
    writer.writerows(cap_gan_trung)

so_nhom = len({hang['group_id'] for hang in hang_nhom})
so_tien_to = sum(hang['derived_by'] == 'filename_prefix' for hang in hang_nhom)
so_du_phong = len(hang_nhom) - so_tien_to
print(f'Đã ghi {len(hang_nhom):,} ảnh vào {DUONG_DAN_NHOM}.')
print(f'Đã ghi {len(cap_gan_trung):,} cặp vào {DUONG_DAN_CAP}.')
print(f'Có {so_nhom:,} group_id; {so_tien_to:,} ảnh theo tiền tố tên và {so_du_phong:,} ảnh theo cụm phash.')


Đã ghi 4,837 ảnh vào /Users/Project/kltn-pcb/data/splits/groups.csv.
Đã ghi 217,179 cặp vào /Users/Project/kltn-pcb/results/pku_near_duplicate_pairs.csv.
Có 10 group_id; 4,837 ảnh theo tiền tố tên và 0 ảnh theo cụm phash.


## Kiểm tra artifact và ghi audit

Các assert dưới đây xác nhận đủ mọi ảnh, không có đường dẫn lặp, thứ tự cột chính xác, đường dẫn đều tương đối và chỉ có hai giá trị nguồn suy luận hợp lệ. `docs/data_audit.md` được tạo đúng một đoạn tiếng Việt theo ba vế bắt buộc, đồng thời chèn các số đếm từ lần thực thi hiện tại.

In [5]:
with DUONG_DAN_NHOM.open(newline='', encoding='utf-8') as tep_csv:
    doc_nhom = list(csv.DictReader(tep_csv))

with DUONG_DAN_NHOM.open(newline='', encoding='utf-8') as tep_csv:
    doc = csv.DictReader(tep_csv)
    assert doc.fieldnames == COT_NHOM

assert len(doc_nhom) == len(duong_dan_anh)
assert len({hang['image_path'] for hang in doc_nhom}) == len(doc_nhom)
assert all(not Path(hang['image_path']).is_absolute() for hang in doc_nhom)
assert {hang['derived_by'] for hang in doc_nhom} <= {'filename_prefix', 'phash_cluster'}
assert all(len(hang['phash']) == 16 for hang in doc_nhom)

noi_dung_audit = (
    f'group inferred by tiền tố số đầu tiên của tên tệp cho {so_tien_to:,}/{len(hang_nhom):,} ảnh PKU và bằng thành phần liên thông phash cho {so_du_phong:,} ảnh không có tiền tố, tạo {so_nhom:,} group_id; validated by tính imagehash.phash cho toàn bộ {len(hang_nhom):,} ảnh, so sánh mọi cặp và ghi nhận {len(cap_gan_trung):,} cặp có khoảng cách Hamming <= {NGUONG_HAMMING}, đồng thời kiểm tra mỗi đường dẫn xuất hiện đúng một lần và CSV có đúng các cột yêu cầu; limitation phash gần nhau là tín hiệu ứng viên chứ không chứng minh ảnh trùng về ngữ nghĩa, còn tiền tố tên tệp là quy ước dữ liệu cần được rà soát bằng mắt trước khi dùng để chống rò rỉ.'
)
DUONG_DAN_AUDIT.write_text(noi_dung_audit + '\n', encoding='utf-8')

print(f'Xác minh thành công: {len(doc_nhom):,} ảnh duy nhất, {len(cap_gan_trung):,} cặp gần trùng.')
print(f'Đã ghi audit một đoạn vào {DUONG_DAN_AUDIT}.')


Xác minh thành công: 4,837 ảnh duy nhất, 217,179 cặp gần trùng.
Đã ghi audit một đoạn vào /Users/Project/kltn-pcb/docs/data_audit.md.
